# OnT quickstart: train, then score

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HuiYang1997/OnT/blob/main/notebooks/OnT_Quickstart.ipynb)

**OnT** (Ontology Transformer, ISWC 2025) turns an ontology into embeddings: each concept becomes a point in a Poincaré ball.
The core is two calls:

1. `OntologyTransformer.fit(...)` trains a language model on your ontology;
2. `model.score_hierarchy(C, D, ...)` scores an axiom $C \sqsubseteq D$.

Runs on the free Colab **CPU**. Choose **Runtime → Run all**.

## Setup · run once

On Colab this cell installs Java (needed to read OWL files) and OnT. On Linux:
`sudo apt install openjdk-17-jre` and `pip install ontology-transformer==0.1.8`.

In [1]:
import os, sys, subprocess
if "google.colab" in sys.modules:          # Java is not preinstalled on Colab
    subprocess.run("apt-get -qq install -y openjdk-17-jre-headless", shell=True)
    os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

%pip install -q ontology-transformer==0.1.8

import logging, warnings           # keep the output short
warnings.filterwarnings("ignore"); logging.disable(logging.WARNING)

## The ontology

`family.owl`, the example of this tutorial: 6 concepts, 2 relations, 7 axioms.

$\mathsf{Father} \sqsubseteq \mathsf{Male} \sqcap \mathsf{Parent}$,
$\mathsf{Mother} \sqsubseteq \mathsf{Female} \sqcap \mathsf{Parent}$,
$\mathsf{Male} \sqcap \mathsf{Parent} \sqsubseteq \mathsf{Father}$,
$\mathsf{Female} \sqcap \mathsf{Parent} \sqsubseteq \mathsf{Mother}$,
$\mathsf{Child} \sqsubseteq \exists \mathsf{hasParent}.\mathsf{Mother}$,
$\mathsf{Child} \sqsubseteq \exists \mathsf{hasParent}.\mathsf{Father}$,
$\mathsf{Parent} \sqsubseteq \exists \mathsf{hasChild}.\mathsf{Child}$.

OnT learns from the concept **names** (labels), such as *father* and *parent*.

In [2]:
import urllib.request
URL = "https://raw.githubusercontent.com/HuiYang1997/OnT/main/notebooks/family.owl"
if not os.path.exists("family.owl"):         # or upload your own OWL file
    urllib.request.urlretrieve(URL, "family.owl")

## Core step 1 · Train the model

One call: `fit` turns the axioms into training examples, trains the language model (default `all-MiniLM-L12-v2`)
so that each concept becomes a point in the Poincaré ball, and picks the weight $\lambda$ used for scoring.

In [3]:
from ont import OntologyTransformer

model = OntologyTransformer.fit(
    owl_path="family.owl",          # the ontology
    output_dir="ont_family",        # training data, checkpoints, final model
    num_epochs=200, batch_size=8,   # small ontology: more epochs
    learning_rate=2e-4, seed=2,
    eval_ratio=0.0)                 # train on every axiom
print("best lambda:", model.best_lambda)

best lambda: 0.0


## Core step 2 · Score an axiom $C \sqsubseteq D$

Each concept is a point; its distance to the center is its *depth*. $C \sqsubseteq D$ is likely when $C$ is close to $D$ and deeper than $D$:

$$s(C \sqsubseteq D) = -\big(d_\kappa(x_C, x_D) + \lambda\,(\lVert x_D\rVert_\kappa - \lVert x_C\rVert_\kappa)\big)$$

Higher score = more likely. Scores are plausibility, not proofs.

In [4]:
lam = model.best_lambda
C = model.encode(["father"], convert_to_tensor=True)
D = model.encode(["parent"], convert_to_tensor=True)
s = model.score_hierarchy(C, D, weight=lam)
print(f"score(father ⊑ parent) = {s.item():.2f}")

score(father ⊑ parent) = -8.89


In [5]:
import pandas as pd
def score(m, child, parent):                # the two calls above
    C, D = m.encode([child, parent], convert_to_tensor=True)
    return round(m.score_hierarchy(C, D, weight=lam).item(), 2)

parents = {"parent": "stated", "male": "stated", "mother": "false", "child": "false"}
pd.DataFrame([(f"father ⊑ {p}", s, score(model, "father", p)) for p, s in parents.items()],
             columns=["axiom", "in ontology", "score"])

,axiom,in ontology,score
0,father ⊑ parent,stated,-8.89
1,father ⊑ male,stated,-9.86
2,father ⊑ mother,false,-10.06
3,father ⊑ child,false,-15.39


## Extra · Before vs after training

The original language model, without OnT training: same axioms, same formula, same $\lambda$.

In [6]:
base = OntologyTransformer.from_pretrained("sentence-transformers/all-MiniLM-L12-v2")
pd.DataFrame([(f"father ⊑ {p}", s, score(base, "father", p), score(model, "father", p))
              for p, s in parents.items()], columns=["axiom", "in ontology", "before", "after"])

,axiom,in ontology,before,after
0,father ⊑ parent,stated,-6.99,-8.89
1,father ⊑ male,stated,-10.93,-9.86
2,father ⊑ mother,false,-8.27,-10.06
3,father ⊑ child,false,-9.83,-15.39


## Extra · Change the settings

All settings are keyword arguments of `fit`; change them and run the training cell again.
The defaults (1 epoch, batch 64, learning rate 1e-5, `eval_ratio=0.1`) suit large ontologies.

```python
model = OntologyTransformer.fit(
    owl_path="family.owl", output_dir="ont_family",
    base_model="sentence-transformers/all-MiniLM-L12-v2",   # or all-mpnet-base-v2
    num_epochs=200, batch_size=8, learning_rate=2e-4,      # training budget
    eval_ratio=0.0,                  # share of axioms held out to choose λ
    clustering_loss_margin=3.0,      # how close C must be to D
    centripetal_loss_margin=0.5,     # how much further out C must be than D
)
# no training at all: OntologyTransformer.from_pretrained("Hui97/OnT-MiniLM-L12-galen")
```

**Notes.** On a 7-axiom ontology the result depends on the random seed (and can differ between machines); if the ranking looks wrong, try another `seed` in the training cell.
Paper: Yang, Chen, He, Gao & Horrocks. *Language Models as Ontology Encoders*. ISWC 2025. Code: github.com/HuiYang1997/OnT